In [ ]:
# ── Install package from GitHub ──────────────────────────────────────────────
!git clone https://github.com/PneumonAI/pneumonai-ml.git
!pip install -q -e pneumonai-ml/

import sys
sys.path.insert(0, "pneumonai-ml")

# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pneumonai.training.model import build_model
from pneumonai.training.dataset import ChestXrayDataset
from pneumonai.training.trainer import train_one_epoch, validate
from pneumonai.preprocessing.specification import load_preprocessing_spec
from torch.utils.data import DataLoader

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO          = Path("pneumonai-ml")
RSNA_IMAGES   = Path("/kaggle/input/rsna-pneumonia-detection-challenge/stage_2_train_images")
SPLITS_DIR    = Path("/kaggle/input/pneumonai-splits")   # your uploaded split CSVs
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

LOCAL_PREFIX  = "/mnt/c/PneumonAI/pneumonai-ml/data/raw/rsna_pneumonia_2018/train/images"

def remap_paths(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["image_path"] = df["image_path"].str.replace(
        LOCAL_PREFIX, str(RSNA_IMAGES), regex=False
    )
    return df

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
spec   = load_preprocessing_spec(REPO / "configs/preprocessing.yaml")
print(f"Device: {device}")


In [ ]:
train_data = pd.read_csv(SPLITS_DIR/"train.csv")
val_data = pd.read_csv(SPLITS_DIR/"validation.csv")
test_data = pd.read_csv(SPLITS_DIR/"test.csv")
train_data = remap_paths(train_data)
test_data = remap_paths(test_data)
val_data = remap_paths(val_data)
train_dataset = ChestXrayDataset(SPLITS_DIR / "train.csv", spec)
train_dataset.df = train_data  # already remapped above
val_dataset = ChestXrayDataset(SPLITS_DIR / "validation.csv", spec)
val_dataset.df = val_data
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

In [ ]:
CONFIGS = [
    {"arch": "resnet18", "lr": 0.001,  "epochs": 20},
    {"arch": "resnet18", "lr": 0.0001, "epochs": 20},
]

results = {}

for cfg in CONFIGS:
    run_name = f"{cfg['arch']}_lr{cfg['lr']}"
    print(f"\n{'='*50}\nTraining {run_name}\n{'='*50}")

    model = build_model(cfg["arch"], num_classes=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    criterion = torch.nn.BCEWithLogitsLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")

    for epoch in range(cfg["epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{cfg['epochs']} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), CHECKPOINT_DIR / f"{run_name}_best.pt")

    results[run_name] = {"history": history, "best_val_loss": best_val_loss, "config": cfg}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(run_name)
    ax1.plot(history["train_loss"], label="train")
    ax1.plot(history["val_loss"], label="val")
    ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()
    ax2.plot(history["val_acc"], label="val accuracy")
    ax2.set_title("Val Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()
    plt.tight_layout()
    plt.show()

# Summary
print("\n── Results Summary ──")
for name, r in results.items():
    print(f"{name}: best_val_loss={r['best_val_loss']:.4f}, final_val_acc={r['history']['val_acc'][-1]:.4f}")
